# Business Understanding

## Project Background

E-commerce businesses generate large amounts of data through customer orders, payments, product transactions, reviews, and deliveries. Analysing this data can reveal valuable insights into business performance, customer behaviour, operational efficiency, and sales trends.

This project uses the Olist Brazilian E-Commerce Public Dataset to investigate these aspects of an online marketplace and identify opportunities for data-driven business improvement.

## Problem Statement

E-commerce businesses face challenges in understanding the factors influencing sales performance, customer satisfaction, and delivery efficiency. Without effective analysis, identifying patterns in customer behaviour, anticipating delivery delays, and forecasting future demand can be difficult.

This project seeks to address these challenges by analysing historical e-commerce data, investigating business relationships, and developing statistical and predictive models to support informed decision-making.

## Project Objectives

The project aims to:

1. Clean, prepare, and integrate the nine datasets into an analysis-ready format.
2. Explore sales performance, customer behaviour, product trends, and delivery operations.
3. Conduct hypothesis testing to investigate relationships and validate business assumptions.
4. Develop regression models to predict continuous business outcomes.
5. Build classification models to predict categorical outcomes, such as late deliveries.
6. Apply time-series forecasting to estimate future sales and order demand.
7. Evaluate model performance and communicate actionable business insights.

## Key Business Questions

- What factors influence sales performance and order value?
- Which product categories and regions generate the most revenue?
- What factors contribute to delivery delays?
- Is delivery performance associated with customer satisfaction?
- Can order delivery times and values be predicted?
- Can orders at risk of late delivery be identified?
- What sales and order demand patterns can be expected in future periods?

## Expected Outcomes

- A cleaned and integrated dataset.
- Exploratory analysis and visualizations.
- Statistically supported findings.
- Regression and classification models.
- Sales and demand forecasting models.
- Business recommendations based on analytical findings.

## Dataset

**Source:** Olist Brazilian E-Commerce Public Dataset

**Coverage:** Approximately 100,000 orders from 2016 to 2018.

**Data sources:** Nine CSV files covering customers, geolocation, order items, payments, reviews, orders, products, sellers, and product category translations.

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import (StandardScaler,MinMaxScaler,OneHotEncoder,LabelEncoder)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split,cross_val_score,cross_validate,TimeSeriesSplit,GridSearchCV)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report,roc_auc_score,RocCurveDisplay)
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

In [4]:
# load datasets
customers = pd.read_csv('Datasets/olist_customers_dataset.csv')
geolocation = pd.read_csv('Datasets/olist_geolocation_dataset.csv')
order_items = pd.read_csv('Datasets/olist_order_items_dataset.csv')
order_payments = pd.read_csv('Datasets/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('Datasets/olist_order_reviews_dataset.csv')
orders = pd.read_csv('Datasets/olist_orders_dataset.csv')
products = pd.read_csv('Datasets/olist_products_dataset.csv')
sellers = pd.read_csv('Datasets/olist_sellers_dataset.csv')
category_translation = pd.read_csv('Datasets/product_category_name_translation.csv')

In [6]:
datasets = {
    'Customers': customers,
    'Geolocation': geolocation,
    'Order Items': order_items,
    'Order Payments': order_payments,
    'Order Reviews': order_reviews,
    'Orders': orders,
    'Products': products,
    'Sellers': sellers,
    'Category Translation': category_translation
}

In [10]:
# data overview
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"{name.upper()} DATASET")
    print('='*60)
    display(df.head())


CUSTOMERS DATASET


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



GEOLOCATION DATASET


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP



ORDER ITEMS DATASET


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


... (8 outputs left out)


SELLERS DATASET


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP



CATEGORY TRANSLATION DATASET


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [7]:
# dataset dimensions
for name, df in datasets.items():
    print(f"\n{name}")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])


Customers
Rows: 99441
Columns: 5

Geolocation
Rows: 1000163
Columns: 5

Order Items
Rows: 112650
Columns: 7

Order Payments
Rows: 103886
Columns: 5

Order Reviews
Rows: 99224
Columns: 7

Orders
Rows: 99441
Columns: 8

Products
Rows: 32951
Columns: 9

Sellers
Rows: 3095
Columns: 4

Category Translation
Rows: 71
Columns: 2


In [8]:
# data types
for name, df in datasets.items():
    print(f"\n{name} Data Types:")
    print(df.dtypes)


Customers Data Types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Geolocation Data Types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Order Items Data Types:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

Order Payments Data Types:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

Order Reviews Data Types:
review_id                  object
order_id                   object
r

In [9]:
# column names 
for name, df in datasets.items():
    print(f"\n{name} Columns:")
    print(df.columns.tolist())


Customers Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Geolocation Columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Order Items Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Order Payments Columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Order Reviews Columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Orders Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Products Columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photo

In [11]:
# missing values
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())
    print("Total missing values:", df.isnull().sum().sum())


Customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Total missing values: 0

Geolocation
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Total missing values: 0

Order Items
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Total missing values: 0

Order Payments
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
Total missing values: 0

Order Reviews
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247

In [12]:
# duplicates
for name, df in datasets.items():
    print(f"\n{name}")
    print("Duplicate rows:", df.duplicated().sum())


Customers
Duplicate rows: 0

Geolocation
Duplicate rows: 261831

Order Items
Duplicate rows: 0

Order Payments
Duplicate rows: 0

Order Reviews
Duplicate rows: 0

Orders
Duplicate rows: 0

Products
Duplicate rows: 0

Sellers
Duplicate rows: 0

Category Translation
Duplicate rows: 0


In [14]:
# geological duplicates
geolocation[geolocation.duplicated()].head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
79,1050,-23.549854,-46.643139,sao paulo,SP
80,1032,-23.540775,-46.635515,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
86,1048,-23.547449,-46.640169,são paulo,SP


In [15]:
print("Total records:", len(geolocation))
print("Unique ZIP-code prefixes:", geolocation['geolocation_zip_code_prefix'].nunique())
print("Unique coordinate pairs:", geolocation[['geolocation_lat', 'geolocation_lng']].drop_duplicates().shape[0])

Total records: 1000163
Unique ZIP-code prefixes: 19015
Unique coordinate pairs: 718463


In [16]:
# remove duplicates in geological
geolocation = geolocation.drop_duplicates()

print("Geolocation shape after removing duplicates:", geolocation.shape)
print("Remaining duplicate rows:", geolocation.duplicated().sum())

Geolocation shape after removing duplicates: (738332, 5)
Remaining duplicate rows: 0


In [17]:
# Check uniqueness of important identifiers

unique_id_checks = {
    'Customers': (customers, 'customer_id'),
    'Orders': (orders, 'order_id'),
    'Products': (products, 'product_id'),
    'Sellers': (sellers, 'seller_id'),
    'Category Translation': (category_translation, 'product_category_name')
}

for name, (df, column) in unique_id_checks.items():
    print(f"\n{name}")
    print("Unique values:", df[column].nunique())
    print("Total records:", len(df))
    print("Duplicate identifiers:", df[column].duplicated().sum())


Customers
Unique values: 99441
Total records: 99441
Duplicate identifiers: 0

Orders
Unique values: 99441
Total records: 99441
Duplicate identifiers: 0

Products
Unique values: 32951
Total records: 32951
Duplicate identifiers: 0

Sellers
Unique values: 3095
Total records: 3095
Duplicate identifiers: 0

Category Translation
Unique values: 71
Total records: 71
Duplicate identifiers: 0


In [18]:
# Check whether foreign keys match existing primary keys
relationship_checks = {
    'Order Items - Orders': (order_items, 'order_id', orders, 'order_id'),
    'Order Items - Products': (order_items, 'product_id', products, 'product_id'),
    'Order Items - Sellers': (order_items, 'seller_id', sellers, 'seller_id'),
    'Orders - Customers': (orders, 'customer_id', customers, 'customer_id'),
    'Order Payments - Orders': (order_payments, 'order_id', orders, 'order_id'),
    'Order Reviews - Orders': (order_reviews, 'order_id', orders, 'order_id')
}

for name, (child, child_col, parent, parent_col) in relationship_checks.items():
    unmatched = ~child[child_col].isin(parent[parent_col])

    print(f"{name}:")
    print("Unmatched records:", unmatched.sum())
    print()

Order Items - Orders:
Unmatched records: 0

Order Items - Products:
Unmatched records: 0

Order Items - Sellers:
Unmatched records: 0

Orders - Customers:
Unmatched records: 0

Order Payments - Orders:
Unmatched records: 0

Order Reviews - Orders:
Unmatched records: 0



In [19]:
# date columns in each dataset
date_columns = {
    'Order Items': (order_items, ['shipping_limit_date']),
    'Order Reviews': (order_reviews,
        ['review_creation_date', 'review_answer_timestamp']
    ),
    'Orders': (
        orders,
        [
            'order_purchase_timestamp',
            'order_approved_at',
            'order_delivered_carrier_date',
            'order_delivered_customer_date',
            'order_estimated_delivery_date'
        ]
    )
}

for name, (df, columns) in date_columns.items():
    for column in columns:
        df[column] = pd.to_datetime(df[column], errors='coerce')

    print(f"\n{name}")
    print(df[columns].dtypes)


Order Items
shipping_limit_date    datetime64[ns]
dtype: object

Order Reviews
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

Orders
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [20]:
for name, (df, columns) in date_columns.items():
    print(f"\n{name}")

    for column in columns:
        print(f"{column}: {df[column].isna().sum()} missing values")


Order Items
shipping_limit_date: 0 missing values

Order Reviews
review_creation_date: 0 missing values
review_answer_timestamp: 0 missing values

Orders
order_purchase_timestamp: 0 missing values
order_approved_at: 160 missing values
order_delivered_carrier_date: 1783 missing values
order_delivered_customer_date: 2965 missing values
order_estimated_delivery_date: 0 missing values


In [21]:
# Examine order status distribution
print(orders['order_status'].value_counts())

# Investigate missing timestamps by order status
missing_by_status = orders.groupby('order_status')[
    [
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date'
    ]
].apply(lambda x: x.isna().sum())

missing_by_status

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [22]:
# delivered orders with missing timestamps
delivered_missing = orders[
    (orders['order_status'] == 'delivered') &
    (
        orders['order_approved_at'].isna() |
        orders['order_delivered_carrier_date'].isna() |
        orders['order_delivered_customer_date'].isna()
    )
]

print("Delivered orders with missing timestamps:", len(delivered_missing))

delivered_missing[
    [
        'order_id',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
]

Delivered orders with missing timestamps: 23


,order_id,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
5323,e04abd8149ef81b95221e88f6ed9ab6a,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
22663,5cf925b116421afa85ee25e99b4c34fb,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22


In [23]:
# numerical columns across datasets

for name, df in datasets.items():
    numeric_df = df.select_dtypes(include=np.number)

    if not numeric_df.empty:
        print(f"\n{name}")
        display(numeric_df.describe().T)


Customers


,count,mean,std,min,25%,50%,75%,max
customer_zip_code_prefix,99441.0,35137.474583,29797.938996,1003.0,11347.0,24416.0,58900.0,99990.0



Geolocation


,count,mean,std,min,25%,50%,75%,max
geolocation_zip_code_prefix,1000163.0,36574.166466,30549.335710,1001.000000,11075.000000,26530.000000,63504.000000,99990.000000
geolocation_lat,1000163.0,-21.176153,5.715866,-36.605374,-23.603546,-22.919377,-19.979620,45.065933
geolocation_lng,1000163.0,-46.390541,4.269748,-101.466766,-48.573172,-46.637879,-43.767709,121.105394



Order Items


,count,mean,std,min,25%,50%,75%,max
order_item_id,112650.0,1.197834,0.705124,1.00,1.00,1.00,1.00,21.00
price,112650.0,120.653739,183.633928,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.990320,15.806405,0.00,13.08,16.26,21.15,409.68


... (4 outputs left out)


Products


,count,mean,std,min,25%,50%,75%,max
product_name_lenght,32341.0,48.476949,10.245741,5.0,42.0,51.0,57.0,76.0
product_description_lenght,32341.0,771.495285,635.115225,4.0,339.0,595.0,972.0,3992.0
product_photos_qty,32341.0,2.188986,1.736766,1.0,1.0,1.0,3.0,20.0
product_weight_g,32949.0,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,30.815078,16.914458,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,16.937661,13.637554,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,23.196728,12.079047,6.0,15.0,20.0,30.0,118.0



Sellers


,count,mean,std,min,25%,50%,75%,max
seller_zip_code_prefix,3095.0,32291.059451,32713.45383,1001.0,7093.5,14940.0,64552.5,99730.0


In [24]:
# geographically unusual coordinates
invalid_coordinates = geolocation[
    (geolocation['geolocation_lat'] < -34) |
    (geolocation['geolocation_lat'] > 6) |
    (geolocation['geolocation_lng'] < -74) |
    (geolocation['geolocation_lng'] > -34)
]

print("Records outside approximate Brazil boundaries:", len(invalid_coordinates))

display(invalid_coordinates.head(10))

Records outside approximate Brazil boundaries: 33


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [25]:
# Check zero and negative values in important numerical columns
numerical_checks = {
    'Order Items': (order_items, ['price', 'freight_value']),
    'Order Payments': (order_payments, ['payment_value', 'payment_installments']),
    'Products': (
        products,
        [
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    )
}

for name, (df, columns) in numerical_checks.items():
    print(f"\n{name}")

    for column in columns:
        print(
            f"{column}: "
            f"Zero = {(df[column] == 0).sum()}, "
            f"Negative = {(df[column] < 0).sum()}"
        )


Order Items
price: Zero = 0, Negative = 0
freight_value: Zero = 383, Negative = 0

Order Payments
payment_value: Zero = 9, Negative = 0
payment_installments: Zero = 2, Negative = 0

Products
product_weight_g: Zero = 4, Negative = 0
product_length_cm: Zero = 0, Negative = 0
product_height_cm: Zero = 0, Negative = 0
product_width_cm: Zero = 0, Negative = 0


In [26]:
# Inspect zero payment values
zero_payments = order_payments[order_payments['payment_value'] == 0]

display(zero_payments)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [27]:
# Inspect zero payment installments
zero_installments = order_payments[order_payments['payment_installments'] == 0]

display(zero_installments)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [28]:
zero_weight = products[products['product_weight_g'] == 0]

display(zero_weight)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [29]:
# Inspect products with missing information
missing_products = products[products.isna().any(axis=1)]

print("Products with missing values:", len(missing_products))

display(missing_products.head(10))

Products with missing values: 611


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [30]:
# Count missing values among affected products

missing_products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [31]:
missing_dimensions = products[
    products[
        [
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    ].isna().any(axis=1)
]

display(missing_dimensions)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
# Examine review comment availability

print("Total reviews:", len(order_reviews))

print("\nMissing review titles:")
print(order_reviews['review_comment_title'].isna().sum())

print("\nMissing review messages:")
print(order_reviews['review_comment_message'].isna().sum())

print("\nBoth title and message missing:")
print(
    order_reviews[
        order_reviews['review_comment_title'].isna() &
        order_reviews['review_comment_message'].isna()
    ].shape[0]
)

Total reviews: 99224

Missing review titles:
87656

Missing review messages:
58247

Both title and message missing:
56518


In [33]:
# Calculate order status percentages
status_summary = orders['order_status'].value_counts().to_frame('Count')

status_summary['Percentage'] = (status_summary['Count'] / len(orders) * 100).round(2)

display(status_summary)

,Count,Percentage
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


In [35]:
# Create working copies of all datasets

customers_clean = customers.copy()
geolocation_clean = geolocation.drop_duplicates().copy()
order_items_clean = order_items.copy()
order_payments_clean = order_payments.copy()
order_reviews_clean = order_reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

In [37]:
# Standardize column names across all datasets

clean_datasets = {
    'customers': customers_clean,
    'geolocation': geolocation_clean,
    'order_items': order_items_clean,
    'order_payments': order_payments_clean,
    'order_reviews': order_reviews_clean,
    'orders': orders_clean,
    'products': products_clean,
    'sellers': sellers_clean,
    'category_translation': category_translation_clean
}

for name, df in clean_datasets.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
    )

# Correct spelling inconsistencies
products_clean.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
}, inplace=True)

print(products_clean.columns.tolist())

['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [38]:
# Remove geographically invalid coordinates

invalid_coordinates = (
    (geolocation_clean['geolocation_lat'] < -34) |
    (geolocation_clean['geolocation_lat'] > 6) |
    (geolocation_clean['geolocation_lng'] < -74) |
    (geolocation_clean['geolocation_lng'] > -34)
)

geolocation_clean = geolocation_clean.loc[~invalid_coordinates].copy()

print("Remaining geolocation records:", len(geolocation_clean))
print("Records outside approximate Brazil boundaries:", (
    (geolocation_clean['geolocation_lat'] < -34) |
    (geolocation_clean['geolocation_lat'] > 6) |
    (geolocation_clean['geolocation_lng'] < -74) |
    (geolocation_clean['geolocation_lng'] > -34)
).sum())

Remaining geolocation records: 738299
Records outside approximate Brazil boundaries: 0


In [39]:
# Replace zero product weights with missing values
products_clean['product_weight_g'] = (products_clean['product_weight_g'].replace(0, np.nan))

print("Missing product weights:", products_clean['product_weight_g'].isna().sum())

Missing product weights: 6


In [40]:
# Replace zero installment values with missing values
order_payments_clean['payment_installments'] = (order_payments_clean['payment_installments'].replace(0, np.nan))

print("Missing installment values:",order_payments_clean['payment_installments'].isna().sum())

Missing installment values: 2


In [41]:
print("Duplicate Portuguese categories:",
      category_translation_clean['product_category_name'].duplicated().sum())

print("Missing Portuguese categories:",
      category_translation_clean['product_category_name'].isna().sum())

Duplicate Portuguese categories: 0
Missing Portuguese categories: 0


In [42]:
# merge english category names
products_clean = products_clean.merge(category_translation_clean,on='product_category_name',how='left')

products_clean.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [43]:
unmatched_categories = products_clean[products_clean['product_category_name'].notna() &products_clean['product_category_name_english'].isna()]

print("Products with unmatched categories:", len(unmatched_categories))
print(unmatched_categories['product_category_name'].value_counts())

Products with unmatched categories: 13
product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64


In [44]:
category_fixes = {
    'portateis_cozinha_e_preparadores_de_alimentos':
        'kitchen_portables_and_food_preparers',
    'pc_gamer': 'gaming_pcs'
}

products_clean['product_category_name_english'] = (products_clean['product_category_name_english'].fillna(products_clean['product_category_name'].map(category_fixes)))

print(
    "Remaining unmatched categories:",
    products_clean[products_clean['product_category_name'].notna() &products_clean['product_category_name_english'].isna()
    ].shape[0]
)

Remaining unmatched categories: 0


In [45]:
print("Product dataset shape:", products_clean.shape)

print("\nMissing values:")
print(products_clean.isnull().sum())

print("\nDuplicate product IDs:",
      products_clean['product_id'].duplicated().sum())

Product dataset shape: (32951, 10)

Missing values:
product_id                         0
product_category_name            610
product_name_length              610
product_description_length       610
product_photos_qty               610
product_weight_g                   6
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    610
dtype: int64

Duplicate product IDs: 0


In [46]:
products_clean['product_category_name_english'] = (products_clean['product_category_name_english'].fillna('unknown'))

print(products_clean['product_category_name_english'].isna().sum())

0


In [47]:
print("Orders dataset shape:", orders_clean.shape)

print("\nMissing values:")
print(orders_clean.isnull().sum())

print("\nOrder status counts:")
print(orders_clean['order_status'].value_counts())

Orders dataset shape: (99441, 8)

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [48]:
# Check chronological inconsistencies
invalid_approval = orders_clean[orders_clean['order_approved_at'].notna() &(orders_clean['order_approved_at'] <orders_clean['order_purchase_timestamp'])]

invalid_carrier = orders_clean[orders_clean['order_delivered_carrier_date'].notna() &orders_clean['order_approved_at'].notna() &(orders_clean['order_delivered_carrier_date'] <orders_clean['order_approved_at'])]

invalid_delivery = orders_clean[orders_clean['order_delivered_customer_date'].notna() &orders_clean['order_delivered_carrier_date'].notna() &(orders_clean['order_delivered_customer_date'] <orders_clean['order_delivered_carrier_date'])]

print("Approval before purchase:", len(invalid_approval))
print("Carrier handover before approval:", len(invalid_carrier))
print("Customer delivery before carrier handover:", len(invalid_delivery))

Approval before purchase: 0
Carrier handover before approval: 1359
Customer delivery before carrier handover: 23


In [49]:
invalid_carrier[
    [
        'order_id',
        'order_status',
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date'
    ]
].head(10)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00


In [50]:
print(invalid_carrier['order_status'].value_counts())

order_status
delivered    1350
shipped         9
Name: count, dtype: int64


In [51]:
invalid_delivery[
    [
        'order_id',
        'order_status',
        'order_delivered_carrier_date',
        'order_delivered_customer_date'
    ]
].head(10)

,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01


In [52]:
# Flag carrier handover before approval
orders_clean['invalid_carrier_timestamp'] = (
    orders_clean['order_approved_at'].notna() &
    orders_clean['order_delivered_carrier_date'].notna() &
    (orders_clean['order_delivered_carrier_date'] <
     orders_clean['order_approved_at'])
)

# Flag customer delivery before carrier handover
orders_clean['invalid_delivery_timestamp'] = (
    orders_clean['order_delivered_customer_date'].notna() &
    orders_clean['order_delivered_carrier_date'].notna() &
    (orders_clean['order_delivered_customer_date'] <
     orders_clean['order_delivered_carrier_date'])
)

print("Invalid carrier timestamps:",
      orders_clean['invalid_carrier_timestamp'].sum())

print("Invalid delivery timestamps:",
      orders_clean['invalid_delivery_timestamp'].sum())

Invalid carrier timestamps: 1359
Invalid delivery timestamps: 23


In [53]:
# Select delivered orders with valid delivery timestamps
delivered_orders = orders_clean[
    (orders_clean['order_status'] == 'delivered') &
    orders_clean['order_delivered_customer_date'].notna() &
    orders_clean['order_delivered_carrier_date'].notna() &
    ~orders_clean['invalid_carrier_timestamp'] &
    ~orders_clean['invalid_delivery_timestamp']
].copy()

print("Valid delivered orders:", delivered_orders.shape[0])
print("Remaining invalid carrier timestamps:",
      delivered_orders['invalid_carrier_timestamp'].sum())
print("Remaining invalid delivery timestamps:",
      delivered_orders['invalid_delivery_timestamp'].sum())

Valid delivered orders: 95096
Remaining invalid carrier timestamps: 0
Remaining invalid delivery timestamps: 0


In [54]:
delivered_orders['delivery_duration_days'] = (delivered_orders['order_delivered_customer_date'] -delivered_orders['order_purchase_timestamp']).dt.total_seconds() / 86400

print(delivered_orders['delivery_duration_days'].describe())

count    95096.000000
mean        12.618067
std          9.570999
min          0.533414
25%          6.812451
50%         10.271759
75%         15.808032
max        209.628611
Name: delivery_duration_days, dtype: float64


In [55]:
# create a binary target for classification
delivered_orders['late_delivery'] = (delivered_orders['order_delivered_customer_date'] >delivered_orders['order_estimated_delivery_date']).astype(int)

print(delivered_orders['late_delivery'].value_counts())
print("\nPercentages:")
print(delivered_orders['late_delivery'].value_counts(normalize=True).mul(100))

late_delivery
0    87304
1     7792
Name: count, dtype: int64

Percentages:
late_delivery
0    91.806175
1     8.193825
Name: proportion, dtype: float64


In [56]:
print("Order items shape:", order_items_clean.shape)

print("\nMissing values:")
print(order_items_clean.isnull().sum())

print("\nDuplicate rows:", order_items_clean.duplicated().sum())

print("\nUnique orders:", order_items_clean['order_id'].nunique())

Order items shape: (112650, 7)

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Duplicate rows: 0

Unique orders: 98666


In [58]:
# Order Item Values
order_items_clean[['order_item_id', 'price', 'freight_value']].describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


In [60]:
# Order-Level Sales Metrics
order_item_summary = (order_items_clean.groupby('order_id').agg(total_order_value=('price', 'sum'),total_freight=('freight_value', 'sum'),total_items=('order_item_id', 'count'))
    .reset_index()
)

print("Order-level dataset shape:", order_item_summary.shape)
order_item_summary.head()

Order-level dataset shape: (98666, 4)


,order_id,total_order_value,total_freight,total_items
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1


In [61]:
print("Payments dataset shape:", order_payments_clean.shape)

print("\nMissing values:")
print(order_payments_clean.isnull().sum())

print("\nDuplicate rows:", order_payments_clean.duplicated().sum())

print("\nPayment types:")
print(order_payments_clean['payment_type'].value_counts())

Payments dataset shape: (103886, 5)

Missing values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    2
payment_value           0
dtype: int64

Duplicate rows: 0

Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [62]:
print("Payment value statistics:")
display(order_payments_clean['payment_value'].describe())

print("\nInstallment statistics:")
display(order_payments_clean['payment_installments'].describe())

Payment value statistics:


count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64


Installment statistics:


count    103884.000000
mean          2.853404
std           2.687047
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

In [63]:
order_payment_summary = (order_payments_clean.groupby('order_id').agg(
        total_payment=('payment_value', 'sum'),
        payment_records=('payment_sequential', 'count'),
        payment_methods=('payment_type', 'nunique'),
        max_installments=('payment_installments', 'max')
    )
    .reset_index()
)

print("Payment summary shape:", order_payment_summary.shape)

order_payment_summary.head()

Payment summary shape: (99440, 5)


,order_id,total_payment,payment_records,payment_methods,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3.0
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3.0


In [64]:
# Compare Order Values with Payments
payment_comparison = order_item_summary.merge(order_payment_summary,on='order_id',how='inner')

payment_comparison['expected_total'] = (payment_comparison['total_order_value'] +payment_comparison['total_freight'])

payment_comparison['payment_difference'] = (payment_comparison['total_payment'] -payment_comparison['expected_total'])

print("Orders compared:", len(payment_comparison))

print("\nPayment difference statistics:")
print(payment_comparison['payment_difference'].describe())

print("\nOrders with matching totals:",
      np.isclose(
          payment_comparison['total_payment'],
          payment_comparison['expected_total'],
          atol=0.01
      ).sum())

Orders compared: 98665

Payment difference statistics:
count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_difference, dtype: float64

Orders with matching totals: 98365


In [65]:
# payment Discrepancies
payment_comparison.loc[
    payment_comparison['payment_difference'].abs().nlargest(10).index,
    [
        'order_id',
        'total_order_value',
        'total_freight',
        'expected_total',
        'total_payment',
        'payment_difference'
    ]
]

,order_id,total_order_value,total_freight,expected_total,total_payment,payment_difference
79537,ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1403.66,1586.47,182.81
42515,6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,287.91,406.92,119.01
43434,70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,466.93,578.82,111.89
58752,996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,587.90,664.43,76.53
43437,70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,213.15,274.84,61.69
72496,bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,242.01,303.02,61.01
67466,af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,413.17,466.97,53.80
14628,262118ce178bb3e4590a3adcf6d62e6b,119.80,57.94,177.74,126.12,-51.62
73899,bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,348.93,394.36,45.43
54305,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,254.45,293.89,39.44


In [66]:
order_payments_clean[
    order_payments_clean['order_id'] ==
    'ce6d150fb29ada17d2082f4847107665'
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
359,ce6d150fb29ada17d2082f4847107665,1,credit_card,10.0,1586.47


In [67]:
order_items_clean[
    order_items_clean['order_id'] ==
    'ce6d150fb29ada17d2082f4847107665'
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
90875,ce6d150fb29ada17d2082f4847107665,1,638bbb2a5e4f360b71f332ddfebfd672,c4af86330efa7a2620772227d2d670c9,2018-02-07 12:31:37,1299.0,104.66


In [68]:
matching_orders = np.isclose(
    payment_comparison['total_payment'],
    payment_comparison['expected_total'],
    atol=0.01
)

print("Payment reconciliation summary")
print("Orders compared:", len(payment_comparison))
print("Matching orders:", matching_orders.sum())
print("Non-matching orders:", (~matching_orders).sum())
print(
    "Matching percentage:",
    round(matching_orders.mean() * 100, 2), "%"
)

Payment reconciliation summary
Orders compared: 98665
Matching orders: 98365
Non-matching orders: 300
Matching percentage: 99.7 %


In [69]:
print("Reviews dataset shape:", order_reviews_clean.shape)

print("\nMissing values:")
print(order_reviews_clean.isnull().sum())

print("\nDuplicate rows:", order_reviews_clean.duplicated().sum())

print("\nReview score distribution:")
print(order_reviews_clean['review_score'].value_counts().sort_index())

Reviews dataset shape: (99224, 7)

Missing values:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Duplicate rows: 0

Review score distribution:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [70]:
print("Minimum review score:",
      order_reviews_clean['review_score'].min())

print("Maximum review score:",
      order_reviews_clean['review_score'].max())

print("Invalid review scores:",
      (~order_reviews_clean['review_score'].between(1, 5)).sum())

Minimum review score: 1
Maximum review score: 5
Invalid review scores: 0


In [71]:
reviews_per_order = order_reviews_clean.groupby('order_id').size()

print("Unique reviewed orders:", reviews_per_order.size)
print("Orders with multiple reviews:", (reviews_per_order > 1).sum())
print("\nReviews per order:")
print(reviews_per_order.value_counts().sort_index())

Unique reviewed orders: 98673
Orders with multiple reviews: 547

Reviews per order:
1    98126
2      543
3        4
Name: count, dtype: int64


In [72]:
order_review_summary = (order_reviews_clean.groupby('order_id').agg(average_review_score=('review_score', 'mean'),review_count=('review_score', 'count'))
    .reset_index()
)

print("Review summary shape:", order_review_summary.shape)

order_review_summary.head()

Review summary shape: (98673, 3)


,order_id,average_review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [73]:
print("Customers dataset shape:", customers_clean.shape)

print("\nMissing values:")
print(customers_clean.isnull().sum())

print("\nDuplicate customer IDs:",
      customers_clean['customer_id'].duplicated().sum())

print("\nUnique customer IDs:",
      customers_clean['customer_id'].nunique())

print("\nUnique customer identities:",
      customers_clean['customer_unique_id'].nunique())

Customers dataset shape: (99441, 5)

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate customer IDs: 0

Unique customer IDs: 99441

Unique customer identities: 96096


In [74]:
print("Sellers dataset shape:", sellers_clean.shape)

print("\nMissing values:")
print(sellers_clean.isnull().sum())

print("\nDuplicate seller IDs:",
      sellers_clean['seller_id'].duplicated().sum())

print("\nUnique sellers:",
      sellers_clean['seller_id'].nunique())

print("\nUnique seller states:",
      sellers_clean['seller_state'].nunique())

Sellers dataset shape: (3095, 4)

Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Duplicate seller IDs: 0

Unique sellers: 3095

Unique seller states: 23
